# Context Metrics

This notebook creates sub-national context metrics for the national tool.

It is designed to grow section by section. Each section should create a tidy metrics table keyed by `adm_id`. The final section combines all available context metrics and exports one CSV.

## 0. Setup

Set the country, administrative level, and input/output paths here. Country folders use ISO3 codes.

In [41]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.transform import xy
from rasterstats import zonal_stats

pd.set_option("display.max_columns", 100)

In [42]:
# Core parameters
COUNTRY_ISO3 = "KEN"
COUNTRY_NAME = "Kenya"
ADMIN_LEVEL = "adm1"  # options currently available: adm0, adm1, adm2
MODULE = "context"
HAZARD = "none"
WORLDPOP_YEAR = 2025

# Resolve paths whether the notebook is run from the repo root or notebooks/.
NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = NOTEBOOK_CWD.parent if NOTEBOOK_CWD.name == "notebooks" else NOTEBOOK_CWD

BOUNDARY_PATH = REPO_ROOT / "data" / "boundaries" / COUNTRY_ISO3 / ADMIN_LEVEL / f"{COUNTRY_ISO3}_{ADMIN_LEVEL}.shp"
WORLDPOP_DIR = REPO_ROOT / "data" / "raw" / COUNTRY_ISO3 / MODULE / "worldpop"
RWI_DIR = REPO_ROOT / "data" / "raw" / COUNTRY_ISO3 / MODULE / "rwi"
CAPITAL_STOCK_DIR = REPO_ROOT / "data" / "raw" / COUNTRY_ISO3 / MODULE / "capital_stock"
OUTPUT_DIR = REPO_ROOT / "results" / COUNTRY_ISO3 / MODULE
OUTPUT_PATH = OUTPUT_DIR / f"{COUNTRY_ISO3}_{ADMIN_LEVEL}_{MODULE}_metrics.csv"

BOUNDARY_PATH, WORLDPOP_DIR, OUTPUT_PATH

(WindowsPath('C:/Users/Mark.DESKTOP-UFHIN6T/Projects/national_tool_metrics/data/boundaries/KEN/adm1/KEN_adm1.shp'),
 WindowsPath('C:/Users/Mark.DESKTOP-UFHIN6T/Projects/national_tool_metrics/data/raw/KEN/context/worldpop'),
 WindowsPath('C:/Users/Mark.DESKTOP-UFHIN6T/Projects/national_tool_metrics/results/KEN/context/KEN_adm1_context_metrics.csv'))

## 0.1 Load Administrative Boundaries

The current Kenya boundary files use `shapeID` as the stable administrative identifier and `shapeName` as the display name.

In [43]:
ADMIN_ID_FIELD = "shapeID"
ADMIN_NAME_FIELD = "shapeName"

boundaries = gpd.read_file(BOUNDARY_PATH)

required_boundary_cols = {ADMIN_ID_FIELD, ADMIN_NAME_FIELD, "geometry"}
missing_boundary_cols = required_boundary_cols.difference(boundaries.columns)
if missing_boundary_cols:
    raise ValueError(f"Boundary layer is missing required columns: {sorted(missing_boundary_cols)}")

admin_regions = boundaries[[ADMIN_ID_FIELD, ADMIN_NAME_FIELD, "geometry"]].copy()
admin_regions = admin_regions.rename(columns={ADMIN_ID_FIELD: "adm_id", ADMIN_NAME_FIELD: "adm_name"})
admin_regions["country_iso3"] = COUNTRY_ISO3
admin_regions["country_name"] = COUNTRY_NAME
admin_regions["admin_level"] = ADMIN_LEVEL.upper()
admin_regions["module"] = MODULE
admin_regions["hazard"] = HAZARD

admin_regions.head()

,adm_id,adm_name,geometry,country_iso3,country_name,admin_level,module,hazard
0,32016919B72266624462344,Turkana,"POLYGON ((35.94724 4.63093, 35.94857 4.62942, ...",KEN,Kenya,ADM1,context,none
1,32016919B63496705134089,Marsabit,"POLYGON ((36.71131 4.44592, 36.72669 4.44664, ...",KEN,Kenya,ADM1,context,none
2,32016919B2031803566233,Mandera,"POLYGON ((39.77523 3.6681, 39.86773 3.8675, 39...",KEN,Kenya,ADM1,context,none
3,32016919B89873713911655,Wajir,"POLYGON ((39.31809 3.47197, 39.33258 3.46684, ...",KEN,Kenya,ADM1,context,none
4,32016919B96045830258165,West Pokot,"POLYGON ((34.93152 2.43647, 34.93121 2.43842, ...",KEN,Kenya,ADM1,context,none


## 0.2 Helper Functions

The population rasters are summarized using zonal sums. By default, `all_touched=False`, so raster cells are included when their centre falls inside each administrative region. This is a simple and reproducible first pass.

In [44]:
def check_path_exists(path: Path, label: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")


def percentage(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    """Return numerator / denominator * 100, with missing values where denominator is zero."""
    denominator = denominator.replace({0: np.nan})
    return numerator / denominator * 100


def zonal_raster_sum(raster_path: Path, polygons: gpd.GeoDataFrame, all_touched: bool = False) -> pd.Series:
    """Sum raster values within each polygon and return a Series aligned to polygons."""
    check_path_exists(raster_path, "Raster")
    with rasterio.open(raster_path) as src:
        raster_crs = src.crs
        nodata = src.nodata

    polygons_for_stats = polygons
    if raster_crs is not None and polygons.crs != raster_crs:
        polygons_for_stats = polygons.to_crs(raster_crs)

    stats = zonal_stats(
        polygons_for_stats,
        str(raster_path),
        stats=["sum"],
        nodata=nodata,
        all_touched=all_touched,
    )
    return pd.Series([item.get("sum", 0) or 0 for item in stats], index=polygons.index, dtype="float64")


def parse_worldpop_age_sex_rasters(worldpop_dir: Path, country_iso3: str, year: int) -> pd.DataFrame:
    """Create an inventory of WorldPop age-sex raster files."""
    country_lower = country_iso3.lower()
    pattern = re.compile(rf"^{country_lower}_(?P<sex>[fmt])_(?P<age>\d{{2}})_{year}_.*\.tif$", re.IGNORECASE)
    records = []

    for path in sorted(worldpop_dir.glob("*.tif")):
        match = pattern.match(path.name)
        if match:
            records.append(
                {
                    "sex": match.group("sex").lower(),
                    "age_start": int(match.group("age")),
                    "age_code": match.group("age"),
                    "path": path,
                }
            )

    inventory = pd.DataFrame.from_records(records)
    if inventory.empty:
        raise FileNotFoundError(f"No WorldPop age-sex rasters found in {worldpop_dir}")
    return inventory.sort_values(["sex", "age_start"]).reset_index(drop=True)


def find_total_population_raster(worldpop_dir: Path, country_iso3: str, year: int) -> Path:
    country_lower = country_iso3.lower()
    matches = sorted(worldpop_dir.glob(f"{country_lower}_pop_{year}_*.tif"))
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected one total population raster, found {len(matches)} in {worldpop_dir}")
    return matches[0]


def sum_age_group(
    inventory: pd.DataFrame,
    sex: str,
    age_starts: list[int],
    polygons: gpd.GeoDataFrame,
) -> pd.Series:
    """Sum a set of WorldPop age-band rasters for one sex category."""
    selected = inventory[(inventory["sex"] == sex) & (inventory["age_start"].isin(age_starts))]
    found_ages = set(selected["age_start"])
    missing_ages = sorted(set(age_starts).difference(found_ages))
    if missing_ages:
        raise FileNotFoundError(f"Missing WorldPop rasters for sex={sex}, ages={missing_ages}")

    total = pd.Series(0.0, index=polygons.index)
    for raster_path in selected["path"]:
        total = total + zonal_raster_sum(raster_path, polygons)
    return total

## 1. Population And Demographics

This section summarizes WorldPop population rasters to the selected administrative level.

Metrics being reported:

- Total population.
- Female and male population counts.
- Children under 5.
- School age children (5-14)
- Working-age population aged 15-64.
- Female population aged 15-49.
- Older population aged 65+.

WorldPop age-band naming uses `00` for under 1, `01` for ages 1-4, then 5-year age bands from `05` to `90`.

In [45]:
check_path_exists(BOUNDARY_PATH, "Boundary layer")
check_path_exists(WORLDPOP_DIR, "WorldPop directory")

worldpop_inventory = parse_worldpop_age_sex_rasters(WORLDPOP_DIR, COUNTRY_ISO3, WORLDPOP_YEAR)
total_population_raster = find_total_population_raster(WORLDPOP_DIR, COUNTRY_ISO3, WORLDPOP_YEAR)

worldpop_inventory.groupby("sex")["age_start"].apply(list), total_population_raster.name

(sex
 f    [0, 1, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, ...
 m    [0, 1, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, ...
 t    [0, 1, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, ...
 Name: age_start, dtype: object,
 'ken_pop_2025_CN_1km_R2025A_UA_v1.tif')

In [46]:
ALL_AGE_STARTS = sorted(worldpop_inventory["age_start"].unique().tolist())

AGE_UNDER_5 = [0, 1]
AGE_SCHOOL_CHILDREN_5_14 = [5, 10]
AGE_WORKING_15_64 = list(range(15, 65, 5))
AGE_OLDER_65_PLUS = list(range(65, 95, 5))
AGE_FEMALE_15_49 = list(range(15, 50, 5))

population_metrics = admin_regions[["adm_id"]].copy()

# Total population from the WorldPop total population raster.
population_metrics["pop_total"] = zonal_raster_sum(total_population_raster, admin_regions)

# Sex breakdowns from age-sex rasters.
population_metrics["pop_female"] = sum_age_group(worldpop_inventory, "f", ALL_AGE_STARTS, admin_regions)
population_metrics["pop_male"] = sum_age_group(worldpop_inventory, "m", ALL_AGE_STARTS, admin_regions)

# Age breakdowns from total age-band rasters.
population_metrics["pop_under_5"] = sum_age_group(worldpop_inventory, "t", AGE_UNDER_5, admin_regions)
population_metrics["pop_school_children_5_14"] = sum_age_group(worldpop_inventory, "t", AGE_SCHOOL_CHILDREN_5_14, admin_regions)
population_metrics["pop_working_age_15_64"] = sum_age_group(worldpop_inventory, "t", AGE_WORKING_15_64, admin_regions)
population_metrics["pop_older_65_plus"] = sum_age_group(worldpop_inventory, "t", AGE_OLDER_65_PLUS, admin_regions)

# Additional useful sex-age metrics.
population_metrics["pop_female_childbearing_15_49"] = sum_age_group(worldpop_inventory, "f", AGE_FEMALE_15_49, admin_regions)

population_metrics.head()

,adm_id,pop_total,pop_female,pop_male,pop_under_5,pop_school_children_5_14,pop_working_age_15_64,pop_older_65_plus,pop_female_childbearing_15_49
0,32016919B72266624462344,1.074106e+06,518061.122345,556044.553513,151482.689453,305022.468750,596109.131836,21491.445099,255517.524414
1,32016919B63496705134089,6.150882e+05,287648.498199,327438.485016,91460.078125,180124.093750,329090.162109,14412.233093,132223.533203
2,32016919B2031803566233,1.032164e+06,512839.462677,519317.328094,192278.976562,345974.343750,482180.638672,11722.385681,215100.789062
3,32016919B89873713911655,9.020463e+05,420830.742584,481212.463654,141367.371094,286673.984375,463422.831055,10578.886322,196937.869141
4,32016919B96045830258165,7.387396e+05,371952.902466,366780.400208,134426.886719,222575.203125,364448.130859,17283.093384,163925.096680


## 2. Relative Wealth Index

This section summarizes the relative wealth index (RWI) to the selected administrative level.

Metrics being reported:

- `raw_rwi`: mean of RWI points inside each administrative region.
- `pop_weighted_rwi`: mean RWI weighted by the 1 km gridded population. Each valid population cell is assigned the nearest RWI point in a projected CRS before aggregating by administrative region.
- `pop_rwi_q1` to `pop_rwi_q5`: number of people in each national population-weighted RWI quintile, where `q1` is the lowest RWI quintile and `q5` is the highest. Population cells without a nearby RWI point are assigned to the quintile matching their admin region's `raw_rwi`.

In [47]:
MAX_RWI_DISTANCE_M = 5_000 # do not assign population cells to RWI points outside of this distance

In [48]:
check_path_exists(RWI_DIR, "RWI directory")

rwi_csv_path = RWI_DIR / f"{COUNTRY_ISO3.lower()}_relative_wealth_index.csv"
rwi_population_raster = find_total_population_raster(RWI_DIR, COUNTRY_ISO3, WORLDPOP_YEAR)

check_path_exists(rwi_csv_path, "RWI CSV")
check_path_exists(rwi_population_raster, "RWI population raster")

# Load and validate the RWI point table.
rwi_table = pd.read_csv(rwi_csv_path)
required_rwi_columns = {"latitude", "longitude", "rwi"}
missing_rwi_columns = required_rwi_columns.difference(rwi_table.columns)
if missing_rwi_columns:
    raise ValueError(f"RWI CSV is missing columns: {sorted(missing_rwi_columns)}")

rwi_table = rwi_table.copy()
rwi_table["rwi"] = pd.to_numeric(rwi_table["rwi"], errors="coerce")
rwi_table = rwi_table.dropna(subset=["latitude", "longitude", "rwi"])

# Convert RWI coordinates into point geometries.
rwi_points = gpd.GeoDataFrame(
    rwi_table,
    geometry=gpd.points_from_xy(rwi_table["longitude"], rwi_table["latitude"]),
    crs="EPSG:4326",
)

rwi_points.shape, rwi_population_raster.name

((29483, 5), 'ken_pop_2025_CN_1km_R2025A_UA_v1.tif')

In [49]:
# Assign RWI points to admin regions for the unweighted mean.
rwi_points_for_admin = rwi_points
if admin_regions.crs is not None and rwi_points.crs != admin_regions.crs:
    rwi_points_for_admin = rwi_points.to_crs(admin_regions.crs)

rwi_admin_join = gpd.sjoin(
    rwi_points_for_admin,
    admin_regions[["adm_id", "geometry"]],
    how="inner",
    predicate="within",
)

raw_rwi_by_admin = rwi_admin_join.groupby("adm_id")["rwi"].mean().rename("raw_rwi")
raw_rwi_by_admin.head()

adm_id
32016919B10079274489584    0.622583
32016919B10848562282373   -0.163839
32016919B11020559484650    0.448233
32016919B1149170155930    -0.093511
32016919B12007380424127    0.075024
Name: raw_rwi, dtype: float64

In [50]:
# Convert valid population raster cells to points at cell centres.
with rasterio.open(rwi_population_raster) as src:
    if src.crs is None:
        raise ValueError(f"Population raster is missing a CRS: {rwi_population_raster}")

    population_array = src.read(1, masked=True)
    population_values = np.asarray(population_array.filled(np.nan), dtype="float64")
    valid_population = np.isfinite(population_values) & (population_values > 0)
    row_idx, col_idx = np.where(valid_population)
    x_coords, y_coords = xy(src.transform, row_idx, col_idx, offset="center")

    population_cells = gpd.GeoDataFrame(
        {
            "cell_id": np.arange(len(row_idx)),
            "population": population_values[row_idx, col_idx],
        },
        geometry=gpd.points_from_xy(x_coords, y_coords),
        crs=src.crs,
    )

# Keep each population cell assigned to the admin region it falls inside.
admin_for_population = admin_regions[["adm_id", "geometry"]]
if admin_for_population.crs != population_cells.crs:
    admin_for_population = admin_for_population.to_crs(population_cells.crs)

population_cells = gpd.sjoin(
    population_cells,
    admin_for_population,
    how="inner",
    predicate="within",
).drop(columns=["index_right"])

# Use a projected CRS so nearest-neighbor distances are in metres.
metric_crs = admin_regions.estimate_utm_crs() or "EPSG:3857"
population_cells_metric = population_cells.to_crs(metric_crs)
rwi_points_metric = rwi_points.to_crs(metric_crs)

# Attach the nearest RWI point to each population cell, then apply the distance cap.
population_rwi = gpd.sjoin_nearest(
    population_cells_metric,
    rwi_points_metric[["rwi", "geometry"]],
    how="left",
    distance_col="rwi_distance_m",
)
population_rwi = population_rwi[population_rwi['rwi_distance_m'] <= MAX_RWI_DISTANCE_M]
population_rwi = population_rwi.sort_values(["cell_id", "rwi_distance_m"]).drop_duplicates("cell_id")
population_rwi = population_rwi.dropna(subset=["rwi"]).copy()
population_rwi["rwi_weighted_population"] = population_rwi["rwi"] * population_rwi["population"]

# Population-weighted RWI: sum(RWI * population) / sum(population).
pop_weighted_rwi_by_admin = population_rwi.groupby("adm_id").agg(
    weighted_rwi_sum=("rwi_weighted_population", "sum"),
    population_with_rwi=("population", "sum"),
)
pop_weighted_rwi_by_admin["pop_weighted_rwi"] = (
    pop_weighted_rwi_by_admin["weighted_rwi_sum"]
    / pop_weighted_rwi_by_admin["population_with_rwi"].replace({0: np.nan})
)
pop_weighted_rwi_by_admin = pop_weighted_rwi_by_admin["pop_weighted_rwi"]

def weighted_quantile(values: pd.Series, weights: pd.Series, quantiles: list[float]) -> np.ndarray:
    """Calculate weighted quantile cutpoints for one-dimensional values."""
    values_array = np.asarray(values, dtype="float64")
    weights_array = np.asarray(weights, dtype="float64")
    quantiles_array = np.asarray(quantiles, dtype="float64")

    valid = np.isfinite(values_array) & np.isfinite(weights_array) & (weights_array > 0)
    values_array = values_array[valid]
    weights_array = weights_array[valid]
    if len(values_array) == 0:
        raise ValueError("Cannot calculate weighted quantiles without valid values and positive weights")

    sort_order = np.argsort(values_array)
    sorted_values = values_array[sort_order]
    sorted_weights = weights_array[sort_order]
    cumulative_weights = np.cumsum(sorted_weights)

    return np.interp(quantiles_array * cumulative_weights[-1], cumulative_weights, sorted_values)


# Define national RWI quintiles using population-weighted cutpoints.
national_rwi_quintile_cutpoints = weighted_quantile(
    population_rwi["rwi"],
    population_rwi["population"],
    [0.2, 0.4, 0.6, 0.8],
)
population_rwi["national_rwi_quintile"] = (
    np.searchsorted(national_rwi_quintile_cutpoints, population_rwi["rwi"], side="right") + 1
)

# Sum people in each national RWI quintile by admin region.
wealth_quintile_population_by_admin = population_rwi.pivot_table(
    index="adm_id",
    columns="national_rwi_quintile",
    values="population",
    aggfunc="sum",
    fill_value=0,
)
wealth_quintile_population_by_admin = wealth_quintile_population_by_admin.reindex(columns=[1, 2, 3, 4, 5], fill_value=0)
wealth_quintile_population_by_admin = wealth_quintile_population_by_admin.rename(
    columns={quintile: f"pop_rwi_q{quintile}" for quintile in range(1, 6)}
)
wealth_quintile_population_by_admin.columns.name = None

# Impute filtered-out population using the admin region's raw RWI quintile.
population_cells_by_admin = population_cells.groupby("adm_id")["population"].sum()
assigned_quintile_population_by_admin = wealth_quintile_population_by_admin.sum(axis=1)
filtered_population_by_admin = (
    population_cells_by_admin
    - assigned_quintile_population_by_admin.reindex(population_cells_by_admin.index, fill_value=0)
).clip(lower=0).rename("filtered_population")
raw_rwi_quintile_by_admin = pd.Series(
    np.searchsorted(national_rwi_quintile_cutpoints, raw_rwi_by_admin, side="right") + 1,
    index=raw_rwi_by_admin.index,
    name="national_rwi_quintile",
)
filtered_population_for_imputation = pd.concat(
    [filtered_population_by_admin, raw_rwi_quintile_by_admin.reindex(filtered_population_by_admin.index)],
    axis=1,
).dropna(subset=["national_rwi_quintile"]).reset_index()
filtered_population_for_imputation["national_rwi_quintile"] = filtered_population_for_imputation[
    "national_rwi_quintile"
].astype(int)

imputed_quintile_population_by_admin = filtered_population_for_imputation.pivot_table(
    index="adm_id",
    columns="national_rwi_quintile",
    values="filtered_population",
    aggfunc="sum",
    fill_value=0,
)
imputed_quintile_population_by_admin = imputed_quintile_population_by_admin.reindex(
    columns=[1, 2, 3, 4, 5],
    fill_value=0,
).rename(
    columns={quintile: f"pop_rwi_q{quintile}" for quintile in range(1, 6)}
)

wealth_quintile_population_by_admin = wealth_quintile_population_by_admin.add(
    imputed_quintile_population_by_admin,
    fill_value=0,
)

pop_weighted_rwi_by_admin.head()

adm_id
32016919B10079274489584    0.988092
32016919B10848562282373   -0.062198
32016919B11020559484650    0.867901
32016919B1149170155930     0.126893
32016919B12007380424127    0.183291
Name: pop_weighted_rwi, dtype: float64

In [51]:
# Combine all RWI metrics into one admin-level table.
rwi_metrics = admin_regions[["adm_id"]].copy()
rwi_metrics = rwi_metrics.merge(raw_rwi_by_admin.reset_index(), on="adm_id", how="left", validate="one_to_one")
rwi_metrics = rwi_metrics.merge(pop_weighted_rwi_by_admin.reset_index(), on="adm_id", how="left", validate="one_to_one")
rwi_metrics = rwi_metrics.merge(wealth_quintile_population_by_admin.reset_index(), on="adm_id", how="left", validate="one_to_one")

rwi_quintile_population_columns = [f"pop_rwi_q{quintile}" for quintile in range(1, 6)]
rwi_metrics[rwi_quintile_population_columns] = rwi_metrics[rwi_quintile_population_columns].fillna(0)

rwi_metrics.head()

,adm_id,raw_rwi,pop_weighted_rwi,pop_rwi_q1,pop_rwi_q2,pop_rwi_q3,pop_rwi_q4,pop_rwi_q5
0,32016919B72266624462344,-0.532833,-0.166135,536482.917057,211814.937524,120171.146480,205637.332408,0.000000
1,32016919B63496705134089,-0.443073,0.109916,231486.061504,133543.910077,94753.658635,78125.800581,77178.816833
2,32016919B2031803566233,-0.707370,-0.030507,436221.999366,278355.598869,85004.199098,106847.452168,125735.144709
3,32016919B89873713911655,-0.632160,-0.094964,447324.578770,113698.148340,229464.573415,64046.330072,47512.659058
4,32016919B96045830258165,-0.449177,-0.276754,452729.535840,156568.248525,74360.473026,55081.399764,0.000000


## 3. Capital Stock

This section summarizes capital stock rasters to the selected administrative level.

Metrics being reported:

- `capstock_non_residential`: total non-residential capital stock.
- `capstock_residential`: total residential capital stock.
- `capstock_infrastructure`: total infrastructure capital stock.

In [52]:
check_path_exists(CAPITAL_STOCK_DIR, "Capital stock directory")

capital_stock_rasters = {
    "capstock_non_residential": CAPITAL_STOCK_DIR / f"{COUNTRY_ISO3}_nres_capstock.tif",
    "capstock_residential": CAPITAL_STOCK_DIR / f"{COUNTRY_ISO3}_res_capstock.tif",
    "capstock_infrastructure": CAPITAL_STOCK_DIR / f"{COUNTRY_ISO3}_inf_capstock.tif",
}

for metric_name, raster_path in capital_stock_rasters.items():
    check_path_exists(raster_path, f"{metric_name} raster")

capital_stock_rasters

{'capstock_non_residential': WindowsPath('C:/Users/Mark.DESKTOP-UFHIN6T/Projects/national_tool_metrics/data/raw/KEN/context/capital_stock/KEN_nres_capstock.tif'),
 'capstock_residential': WindowsPath('C:/Users/Mark.DESKTOP-UFHIN6T/Projects/national_tool_metrics/data/raw/KEN/context/capital_stock/KEN_res_capstock.tif'),
 'capstock_infrastructure': WindowsPath('C:/Users/Mark.DESKTOP-UFHIN6T/Projects/national_tool_metrics/data/raw/KEN/context/capital_stock/KEN_inf_capstock.tif')}

In [53]:
# Sum each capital stock sector within admin boundaries.
capital_stock_metrics = admin_regions[["adm_id"]].copy()

for metric_name, raster_path in capital_stock_rasters.items():
    capital_stock_metrics[metric_name] = zonal_raster_sum(raster_path, admin_regions)

capital_stock_metrics.head()

C:\Users\Mark.DESKTOP-UFHIN6T\anaconda3\envs\tooling\Lib\site-packages\rasterstats\io.py:352: UserWarning: Setting masked to True because dataset mask has been detected
  warnings.warn(
C:\Users\Mark.DESKTOP-UFHIN6T\anaconda3\envs\tooling\Lib\site-packages\rasterstats\io.py:352: UserWarning: Setting masked to True because dataset mask has been detected
  warnings.warn(
C:\Users\Mark.DESKTOP-UFHIN6T\anaconda3\envs\tooling\Lib\site-packages\rasterstats\io.py:352: UserWarning: Setting masked to True because dataset mask has been detected
  warnings.warn(
C:\Users\Mark.DESKTOP-UFHIN6T\anaconda3\envs\tooling\Lib\site-packages\rasterstats\io.py:352: UserWarning: Setting masked to True because dataset mask has been detected
  warnings.warn(
C:\Users\Mark.DESKTOP-UFHIN6T\anaconda3\envs\tooling\Lib\site-packages\rasterstats\io.py:352: UserWarning: Setting masked to True because dataset mask has been detected
  warnings.warn(
C:\Users\Mark.DESKTOP-UFHIN6T\anaconda3\envs\tooling\Lib\site-packages

,adm_id,capstock_non_residential,capstock_residential,capstock_infrastructure
0,32016919B72266624462344,6.243697e+08,543555584.0,1.209564e+09
1,32016919B63496705134089,5.805701e+08,305375136.0,8.446480e+08
2,32016919B2031803566233,2.315542e+08,497537280.0,7.210340e+08
3,32016919B89873713911655,1.320026e+09,509987456.0,1.134630e+09
4,32016919B96045830258165,9.838196e+08,395260736.0,3.641294e+08


In [54]:
# Register each completed section output here.
# Future sections should append their metrics DataFrames to this list.
section_metric_tables = [
    population_metrics,
    rwi_metrics,
    capital_stock_metrics,
]

## 4. Combine And Export Context Metrics

This final section combines all completed context sections and writes one CSV for the selected country and administrative level.

In [55]:
identifier_columns = ["country_iso3", "country_name", "admin_level", "adm_id", "adm_name", "module", "hazard"]
context_metrics = admin_regions[identifier_columns].copy()

for metrics in section_metric_tables:
    if "adm_id" not in metrics.columns:
        raise ValueError("Each section metrics table must include an 'adm_id' column")
    context_metrics = context_metrics.merge(metrics, on="adm_id", how="left", validate="one_to_one")

# Round floating point outputs to keep the CSV readable. Counts remain model estimates, not integers.
numeric_columns = context_metrics.select_dtypes(include="number").columns
context_metrics[numeric_columns] = context_metrics[numeric_columns].round(3)

context_metrics.head()

,country_iso3,country_name,admin_level,adm_id,adm_name,module,hazard,pop_total,pop_female,pop_male,pop_under_5,pop_school_children_5_14,pop_working_age_15_64,pop_older_65_plus,pop_female_childbearing_15_49,raw_rwi,pop_weighted_rwi,pop_rwi_q1,pop_rwi_q2,pop_rwi_q3,pop_rwi_q4,pop_rwi_q5,capstock_non_residential,capstock_residential,capstock_infrastructure
0,KEN,Kenya,ADM1,32016919B72266624462344,Turkana,context,none,1074106.250,518061.122,556044.554,151482.689,305022.469,596109.132,21491.445,255517.524,-0.533,-0.166,536482.917,211814.938,120171.146,205637.332,0.000,6.243697e+08,543555584.0,1.209564e+09
1,KEN,Kenya,ADM1,32016919B63496705134089,Marsabit,context,none,615088.250,287648.498,327438.485,91460.078,180124.094,329090.162,14412.233,132223.533,-0.443,0.110,231486.062,133543.910,94753.659,78125.801,77178.817,5.805701e+08,305375136.0,8.446480e+08
2,KEN,Kenya,ADM1,32016919B2031803566233,Mandera,context,none,1032164.375,512839.463,519317.328,192278.977,345974.344,482180.639,11722.386,215100.789,-0.707,-0.031,436221.999,278355.599,85004.199,106847.452,125735.145,2.315542e+08,497537280.0,7.210340e+08
3,KEN,Kenya,ADM1,32016919B89873713911655,Wajir,context,none,902046.312,420830.743,481212.464,141367.371,286673.984,463422.831,10578.886,196937.869,-0.632,-0.095,447324.579,113698.148,229464.573,64046.330,47512.659,1.320026e+09,509987456.0,1.134630e+09
4,KEN,Kenya,ADM1,32016919B96045830258165,West Pokot,context,none,738739.625,371952.902,366780.400,134426.887,222575.203,364448.131,17283.093,163925.097,-0.449,-0.277,452729.536,156568.249,74360.473,55081.400,0.000,9.838196e+08,395260736.0,3.641294e+08


In [56]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
context_metrics.to_csv(OUTPUT_PATH, index=False)

print(f"Exported {len(context_metrics):,} rows and {len(context_metrics.columns):,} columns to:")
print(OUTPUT_PATH)

Exported 47 rows and 25 columns to:
C:\Users\Mark.DESKTOP-UFHIN6T\Projects\national_tool_metrics\results\KEN\context\KEN_adm1_context_metrics.csv
